# Airport-OCR — VOBL pipeline (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashpatle23/Airport-OCR/blob/feat/airport-ocr-poc/notebooks/Airport_OCR_Colab.ipynb)

Run the aerodrome-chart extraction pipeline end to end on **your** machine's GPU-free Colab runtime:

`PDF (or words dump) -> intake (SHA-256) -> native-text extraction -> normalize -> validate -> JSON / GeoJSON -> map / search`

> **Non-operational / research only.** Nothing here is authoritative aeronautical
> data and must never be used for navigation. Extracted values stay *provisional*
> until the original source, its rights, and qualified aviation review are recorded.
>
> **Source rights:** the VOBL chart is AAI/BIAL copyrighted material. Use it only
> if you are permitted to; this notebook does not grant any rights.

**Case study:** Kempegowda International Airport, Bengaluru — chart `AD 2 VOBL 1-101`.


## 1. Install

Installs PyMuPDF (to read the PDF) and the `airport-ocr` package straight from the repo branch.

In [ ]:
%pip -q install pymupdf
%pip -q install 'git+https://github.com/yashpatle23/Airport-OCR.git@feat/airport-ocr-poc'

import airport_ocr, fitz
print('airport_ocr', airport_ocr.__version__)
print('PyMuPDF (fitz)', fitz.__doc__.strip().splitlines()[0])
print('operational_use =', airport_ocr.OPERATIONAL_USE, '(research only)')

## 2. Provide the source

Upload either:
- the original **`VOBL-ADC.pdf`**, or
- a **PyMuPDF words dump** JSON (`page.get_text("words")` per page).

If you just want to try it without a PDF, run the *"Use the bundled sample"* cell instead.

In [ ]:
from google.colab import files
uploaded = files.upload()          # pick VOBL-ADC.pdf  (or a words .json)
SRC = list(uploaded.keys())[0]
print('using source:', SRC)

**Optional:** skip the upload and use the small real word-sample shipped in the repo.

In [ ]:
# Optional: bundled sample (a real subset of the VOBL words dump)
import urllib.request
SRC = 'vobl-words-sample.json'
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/yashpatle23/Airport-OCR/feat/airport-ocr-poc/examples/vobl-words-sample.json',
    SRC)
print('using bundled sample:', SRC)

## 3. Intake — integrity and provenance (PDF only)

Computes a SHA-256, sniffs the file type, and records a provenance manifest.
It never claims to malware-scan or to grant rights.

In [ ]:
import json
from airport_ocr.intake import intake_file

if SRC.lower().endswith('.pdf'):
    result = intake_file(SRC)
    manifest = result.manifest()
    print(json.dumps(manifest, indent=2))
    print('\nSHA-256:', result.sha256)
else:
    print('Source is a words dump (not a PDF); skipping binary intake.')

## 4. Native-text extraction

Reads the PDF with PyMuPDF (`get_text("words")`) and reconstructs observations:
airport header, ARP, elevation, the runway table, and the **taxiway inventory**
from the legend. Runway holding positions stay blocked (they need vector geometry).

In [ ]:
import fitz, json
from airport_ocr.pdf_words import extract_from_words

if SRC.lower().endswith('.pdf'):
    doc = fitz.open(SRC)
    pages = [{'page': i, 'size': [p.rect.width, p.rect.height], 'words': p.get_text('words')}
             for i, p in enumerate(doc)]
    json.dump(pages, open('vobl_words.json', 'w'))
    dump = pages
    print('pages:', len(pages), '| words on page 0:', len(pages[0]['words']))
else:
    dump = json.load(open(SRC))

observations = extract_from_words(dump, dataset_id='vobl-adc-native-text')
json.dump(observations, open('observations.json', 'w'), ensure_ascii=False, indent=2)

print('ICAO:', observations['airport_icao'])
print('runway pairs:', [r['designator_pair'] for r in observations['runways']])
print('taxiways:', len(observations['taxiways']['features']))
print('holding positions:', observations['runway_holding_positions']['completeness_status'])

## 5. Normalize + validate

Deterministic DMS -> CRS84 conversion, reciprocal-runway / dimension / unit checks,
elevation-conflict preservation, and taxiway-inventory validation. Produces
normalized JSON, RFC 7946 GeoJSON, and a validation report.

In [ ]:
from airport_ocr.pipeline import normalize

normalized, geojson, report = normalize(observations)
json.dump(normalized, open('normalized.json', 'w'), ensure_ascii=False, indent=2)
json.dump(geojson, open('features.geojson', 'w'), ensure_ascii=False, indent=2)
json.dump(report, open('validation-report.json', 'w'), ensure_ascii=False, indent=2)

print('STATUS:', report['status'])
print('counts:', report['counts'], '| failures:', report['failure_count'])
print('\nchecks:')
for c in report['checks']:
    print(f"  [{c['status']}] {c['id']}")

## 6. Inspect the structured result

In [ ]:
a = normalized['airport']
print('Airport :', a['icao'], '-', a['name'])
print('ARP     :', a['arp']['coordinates'], '(lon, lat, CRS84)')
print('Elevation claims (unresolved conflict kept):')
for c in a['elevation']['claims']:
    print('   ', c['source_text'], '->', c['value'], c['unit'])
print('  selected_value:', a['elevation']['selected_value'])

print('\nRunways:')
for r in normalized['runways']:
    for d in r['directions']:
        t = d['threshold']; lon, lat = t['position']['coordinates']
        print(f"  {d['designator']:>3} ({r['designator_pair']})  lat={lat:.6f} lon={lon:.6f}  "
              f"THR {t['elevation']['value']} / TDZ {t['tdz_elevation']['value']} ft")

print('\nTaxiways ({}):'.format(normalized['taxiways'].get('count', 0)))
print('  ', [f['designator'] for f in normalized['taxiways']['features']])
print('\nRunway holding positions:', normalized['runway_holding_positions']['completeness_status'])

## 7. Map (provisional)

Plots the ARP and runway thresholds. The dashed lines are **threshold connectors**,
not surveyed runway extents.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
for f in geojson['features']:
    g = f['geometry']; p = f['properties']
    if g['type'] == 'Point':
        x, y = g['coordinates']
        is_arp = p['feature_type'] == 'aerodrome_reference_point'
        plt.scatter([x], [y], s=80 if is_arp else 35,
                    c='tab:blue' if is_arp else 'tab:green', zorder=3)
        plt.annotate('ARP' if is_arp else p.get('designator', ''), (x, y),
                     textcoords='offset points', xytext=(5, 5), fontsize=9)
    elif g['type'] == 'LineString':
        xs = [c[0] for c in g['coordinates']]; ys = [c[1] for c in g['coordinates']]
        plt.plot(xs, ys, '--', c='gray', zorder=1)

plt.xlabel('longitude'); plt.ylabel('latitude')
plt.title('VOBL ARP + runway thresholds — provisional, non-operational')
plt.grid(True, alpha=0.3); plt.gca().set_aspect('equal', adjustable='datalim')
plt.show()

## 8. Search the GeoJSON projection

In [ ]:
from airport_ocr.search import search_features

print('thresholds :', search_features(geojson, feature_type='runway_threshold')['properties']['match_count'])
print('09L match  :', search_features(geojson, designator='09L')['properties']['match_count'])
bbox = [77.70, 13.19, 77.71, 13.20]   # tight box around the ARP
res = search_features(geojson, bbox=bbox)
print('in bbox    :', res['properties']['match_count'],
      [f['properties']['feature_type'] for f in res['features']])

## 9. Download the outputs

In [ ]:
from google.colab import files
for name in ['observations.json', 'normalized.json', 'features.geojson', 'validation-report.json']:
    files.download(name)

## 10. Optional — vector geometry for holding positions (experimental)

Runway **holding positions** stay blocked because their identity/geometry are not
in the text layer. The marking geometry lives in the PDF's vector drawings. This
cell dumps that layer so the holding-position work can be developed next; it does
**not** yet produce holding positions.

In [ ]:
if SRC.lower().endswith('.pdf'):
    doc = fitz.open(SRC)
    page0 = doc[0]
    drawings = page0.get_drawings()
    print('vector drawing objects on page 0:', len(drawings))
    json.dump({'page': 0, 'drawings_count': len(drawings)}, open('drawings_summary.json', 'w'))
    print('Saved drawings_summary.json (full geometry export is the next step).')
else:
    print('Vector geometry needs the original PDF (a words dump has no drawings).')

## Notes & remaining blockers

- **Non-operational**: research only; not for navigation.
- **Rights**: confirm permission for the AAI/BIAL source before storing/sharing outputs.
- **Holding positions**: blocked pending vector-geometry work (cell 10 starts it).
- **Elevation conflict**: chart `3003 ft` vs indexed eAIP `3001 FT` is preserved, not resolved.

Source & PR: <https://github.com/yashpatle23/Airport-OCR/pull/1>
